In [41]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nusratjahannawal/banglapsg-v2-deduplicated/BanglaPSG_V2_Deduplicated.xlsx


In [42]:
import glob
import pandas as pd

hits = glob.glob("/kaggle/input/datasets/nusratjahannawal/banglapsg-v2-deduplicated/BanglaPSG_V2_Deduplicated.xlsx", recursive=True)
assert len(hits) == 1, f"Expected exactly one BanglaPSG_V2_Deduplicated.xlsx, found: {hits}"
path = hits[0]
print("Using:", path)

df = pd.read_excel(path)
assert df.columns.tolist() == ["No.", "Grievance Text", "Category", "Severity"], df.columns.tolist()
assert len(df) == 5970, len(df)
assert df["Grievance Text"].duplicated().sum() == 0, "Exact duplicate texts found: wrong file?"
df.head()

Using: /kaggle/input/datasets/nusratjahannawal/banglapsg-v2-deduplicated/BanglaPSG_V2_Deduplicated.xlsx


,No.,Grievance Text,Category,Severity
0,1,সরকারি হাসপাতাল গুলো এত অপরিষ্কার আর নোংরা।,Healthcare,High
1,2,মেডিসিন বিশেষজ্ঞ ডাঃ বসে না এসিস্ট্যান্ট দিয়ে...,Healthcare,High
2,3,রূপগঞ্জে মানুষ মারার কারখানা ইউ এস-বাংলা হাসপাতাল,Healthcare,Low
3,4,সরকারি হাসপাতালগুলোর সমস্ত টেস্টের মেশিন গুলো ...,Healthcare,Medium
4,5,সামান্য জ্বর আসলে এক হাজার নয়শত টাকার টেস্ট দয়,Healthcare,Low


In [43]:
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (5970, 4)

Columns:
['No.', 'Grievance Text', 'Category', 'Severity']


In [44]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
No.               0
Grievance Text    0
Category          0
Severity          0
dtype: int64


In [45]:
import re, unicodedata

def norm_key(t):
    t = unicodedata.normalize("NFC", str(t))
    t = re.sub(r"[\u200b\u200c\u200d\u2060\ufeff]", "", t)
    t = t.replace("\\n", " ")
    t = re.sub(r"[^\u0980-\u09FFA-Za-z0-9]+", " ", t)
    return re.sub(r"\s+", " ", t).strip().lower()

# df.duplicated() can never fire here because "No." is unique, so check the TEXT itself
print("Exact duplicate texts:", df["Grievance Text"].duplicated().sum())
print("Normalized duplicate texts:", df["Grievance Text"].map(norm_key).duplicated().sum())

Exact duplicate texts: 0
Normalized duplicate texts: 0


In [46]:
category_count = df["Category"].value_counts()

print(category_count)

Category
Food                1030
Road & Transport    1025
Electricity          995
Waste                992
Healthcare           983
Water                945
Name: count, dtype: int64


In [47]:
severity_count = df["Severity"].value_counts()

print(severity_count)

Severity
Medium    2410
Low       1951
High      1609
Name: count, dtype: int64


In [48]:
category_severity = pd.crosstab(
    df["Category"],
    df["Severity"]
)

category_severity

Severity,High,Low,Medium
Category,,,
Electricity,205,376,414
Food,167,392,471
Healthcare,330,290,363
Road & Transport,371,223,431
Waste,212,358,422
Water,324,312,309


In [49]:
df["text_length_words"] = df["Grievance Text"].astype(str).apply(
    lambda x: len(x.split())
)

print(df["text_length_words"].describe())

count    5970.000000
mean       21.350084
std        13.415586
min         2.000000
25%        12.000000
50%        18.000000
75%        26.000000
max       125.000000
Name: text_length_words, dtype: float64


In [50]:
print("Categories:")
print(df["Category"].unique())

print("\nSeverity:")
print(df["Severity"].unique())

Categories:
['Healthcare' 'Electricity' 'Water' 'Waste' 'Food' 'Road & Transport']

Severity:
['High' 'Low' 'Medium']


In [51]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [52]:
!pip install -q transformers datasets accelerate evaluate sentencepiece

In [53]:
import pandas as pd
import numpy as np
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

In [54]:
data = df[["Grievance Text", "Category"]].copy()

data.columns = ["text", "label"]

label_encoder = LabelEncoder()

data["label"] = label_encoder.fit_transform(data["label"])

print(label_encoder.classes_)

num_labels = len(label_encoder.classes_)

print("Number of classes:", num_labels)

['Electricity' 'Food' 'Healthcare' 'Road & Transport' 'Waste' 'Water']
Number of classes: 6


In [55]:
#Leakage-safe split built from BanglaPSG_V2_Deduplicated.xlsx
import re, unicodedata, hashlib
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

_SEED = 42
_CHAR_COS, _WORD_COS, _CONTAIN, _MIN_WORDS = 0.65, 0.60, 0.90, 4
_RATIO = np.array([0.70, 0.15, 0.15])         # train / val / test

def _norm_key(t):
    t = unicodedata.normalize("NFC", str(t))
    t = re.sub(r"[\u200b\u200c\u200d\u2060\ufeff]", "", t)
    t = t.replace("\\n", " ")
    t = re.sub(r"[^\u0980-\u09FFA-Za-z0-9]+", " ", t)
    return re.sub(r"\s+", " ", t).strip().lower()

_text = df["Grievance Text"].astype(str).map(lambda t: unicodedata.normalize("NFC", t))
_ks = _text.map(_norm_key)
_key = _ks.str.replace(" ", "", regex=False)
_strata = (df["Category"].astype(str) + " | " + df["Severity"].astype(str)).values
_n = len(df)
assert _text.duplicated().sum() == 0 and _key.duplicated().sum() == 0, "Input still has duplicates"

# 1)link near-duplicates
def _pairs(X, thr):
    out = []
    for s in range(0, _n, 500):
        sim = (X[s:s + 500] @ X.T).toarray()
        r, c = np.where(sim >= thr)
        out += [(s + a, b) for a, b in zip(r, c) if b > s + a]
    return out

_Xc = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True).fit_transform(_ks)
_Xw = TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), sublinear_tf=True).fit_transform(_ks)
_links = set(_pairs(_Xc, _CHAR_COS)) | set(_pairs(_Xw, _WORD_COS))

_B = CountVectorizer(token_pattern=r"\S+", binary=True).fit_transform(_ks).tocsr().astype(np.float32)
_nw = np.asarray(_B.sum(axis=1)).ravel()
_inter = (_B @ _B.T).tocoo()
_m = _inter.row < _inter.col
_I, _J, _V = _inter.row[_m], _inter.col[_m], _inter.data[_m]
_mn = np.minimum(_nw[_I], _nw[_J])
_sel = (_mn >= _MIN_WORDS) & (_V / _mn >= _CONTAIN - 1e-9)
_links |= set(zip(_I[_sel].tolist(), _J[_sel].tolist()))

_L = np.array(sorted(_links))
_adj = coo_matrix((np.ones(len(_L)), (_L[:, 0], _L[:, 1])), shape=(_n, _n))
_, _grp = connected_components(_adj, directed=False)
assert np.bincount(_grp).max() <= 10, "A giant linked group formed"

# 2)stratified group assignment: pure numpy + hashing
_sid = np.unique(_strata, return_inverse=True)[1]
_S, _G = _sid.max() + 1, _grp.max() + 1
_comp = np.zeros((_G, _S))
np.add.at(_comp, (_grp, _sid), 1)                      
_first = {}
for _i, _g in enumerate(_grp):
    _first.setdefault(_g, _key.iat[_i])
_tie = np.array([int(hashlib.md5(f"{_SEED}|{_first[_g]}".encode()).hexdigest()[:12], 16) for _g in range(_G)])
_order = np.lexsort((_tie, -_comp.sum(1)))              # big groups first, seeded pseudo-random tie-break
_tgt = _RATIO[:, None] * _comp.sum(0)[None, :]          # target rows per split per stratum
_cur = np.zeros((3, _S))
_assign = np.zeros(_G, dtype=int)
for _g in _order:
    _new = _cur + _comp[_g]
    _cost = (((_new - _tgt) ** 2 - (_cur - _tgt) ** 2) / np.maximum(_tgt, 1)).sum(axis=1)
    _s = int(np.argmin(_cost))
    _assign[_g] = _s
    _cur[_s] = _new[_s]
_split = np.array(["train", "val", "test"])[_assign[_grp]]
df["Group"], df["Split"] = _grp, _split
_dev = np.abs(_cur - _tgt).max()
print("Worst per-stratum deviation from the 70/15/15 target (rows):", round(_dev, 1))
assert _dev <= 4, "Split is not balanced across Category x Severity"

# 3)leakage checks
for _a, _b in [("train", "val"), ("train", "test"), ("val", "test")]:
    _ma, _mb = _split == _a, _split == _b
    assert not set(_text[_ma]) & set(_text[_mb]), f"Exact text overlap {_a}/{_b}"
    assert not set(_key[_ma]) & set(_key[_mb]), f"Normalized text overlap {_a}/{_b}"
    assert not set(_grp[_ma]) & set(_grp[_mb]), f"Near-duplicate group shared by {_a}/{_b}"
assert all(_split[i] == _split[j] for i, j in _links), "A linked pair crosses splits"
assert len(set(_strata)) == len(set(_strata[_split == "train"])) == len(set(_strata[_split == "val"])) == len(set(_strata[_split == "test"]))

# 4)task-specific frames
train_df = data[df["Split"] == "train"].reset_index(drop=True)
valid_df = data[df["Split"] == "val"].reset_index(drop=True)
test_df  = data[df["Split"] == "test"].reset_index(drop=True)
assert len(train_df) + len(valid_df) + len(test_df) == len(data)
assert set(train_df["label"]) == set(valid_df["label"]) == set(test_df["label"])

print("Linked pairs:", len(_links), "| groups with >1 row:", int((np.bincount(_grp) > 1).sum()))
print("Train:", train_df.shape, "| Validation:", valid_df.shape, "| Test:", test_df.shape)
print("Split fingerprint:", hashlib.sha1(",".join(map(str, sorted(df.loc[df["Split"] == "test", "No."]))).encode()).hexdigest()[:12],
      "(must be identical in all 6 notebooks)")
print("NO LEAKAGE: exact text, normalized text and near-duplicate groups are disjoint across splits")

Worst per-stratum deviation from the 70/15/15 target (rows): 0.8
Linked pairs: 94 | groups with >1 row: 68
Train: (4176, 2) | Validation: (899, 2) | Test: (895, 2)
Split fingerprint: 9dc9e17dd435 (must be identical in all 6 notebooks)
NO LEAKAGE: exact text, normalized text and near-duplicate groups are disjoint across splits


In [56]:
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

In [57]:
model_name = "csebuetnlp/banglabert"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [58]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )


train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/4176 [00:00<?, ? examples/s]

Map:   0%|          | 0/899 [00:00<?, ? examples/s]

Map:   0%|          | 0/895 [00:00<?, ? examples/s]

In [59]:
train_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

valid_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

test_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

In [60]:
set_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoi

In [61]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),

        "precision": precision_score(
            labels,
            predictions,
            average="macro"
        ),

        "recall": recall_score(
            labels,
            predictions,
            average="macro"
        ),

        "f1": f1_score(
            labels,
            predictions,
            average="macro"
        )
    }

In [62]:
training_args = TrainingArguments(
    output_dir="./banglabert_category",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    logging_steps=50,

    fp16=True,

    report_to="none",
    seed=42,
    data_seed=42
)

In [63]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [64]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.898804,0.578384,0.952169,0.952813,0.952031,0.952275
2,0.322073,0.395728,0.955506,0.955909,0.955596,0.955605
3,0.176087,0.386465,0.959956,0.960190,0.960027,0.959964


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=783, training_loss=0.8999218654510771, metrics={'train_runtime': 262.088, 'train_samples_per_second': 47.801, 'train_steps_per_second': 2.988, 'total_flos': 824093421133824.0, 'train_loss': 0.8999218654510771, 'epoch': 3.0})

In [65]:
result = trainer.evaluate(test_dataset)

result

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.45055902004241943,
 'eval_accuracy': 0.9486033519553073,
 'eval_precision': 0.9486476194021117,
 'eval_recall': 0.948769825667192,
 'eval_f1': 0.9485249603670657,
 'eval_runtime': 5.3547,
 'eval_samples_per_second': 167.142,
 'eval_steps_per_second': 10.458,
 'epoch': 3.0}

In [66]:
pred = trainer.predict(test_dataset)

y_pred = np.argmax(pred.predictions, axis=1)

y_true = pred.label_ids
assert (y_true == test_df["label"].values).all()

print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


                  precision    recall  f1-score   support

     Electricity       0.95      0.95      0.95       149
            Food       0.98      0.94      0.96       155
      Healthcare       0.95      0.98      0.97       146
Road & Transport       0.95      0.93      0.94       154
           Waste       0.92      0.96      0.94       149
           Water       0.93      0.94      0.93       142

        accuracy                           0.95       895
       macro avg       0.95      0.95      0.95       895
    weighted avg       0.95      0.95      0.95       895



In [67]:
model_save_path = "./BanglaBERT_Category_Model"

trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!
